# Usines et stations : premier regard

In [ ]:
%%capture
# Étapes précédentes : exécutées sans afficher leurs figures ni leurs messages, et une seule fois par noyau
# (si les données ne sont pas encore en cache, exécuter d'abord donnees.ipynb seul pour suivre le téléchargement)
if "ETAPE_01" not in globals():
    %run ./donnees.ipynb
plt.close("all")

### Premier résultat : Répartition des usines et des stations hydrologiques dans le Nord

In [ ]:
# Stations (points bleus, taille réduite, avec un fin contour blanc pour la lisibilité)
ax = plt.subplots(figsize=(12, 10))[1]
installations_indust.to_crs(epsg=3857).plot(ax=ax, color="darkred", markersize=8, label="Usines ICPE", zorder=1)
stations.to_crs(epsg=3857).plot(ax=ax, color="blue", markersize=8, label="Stations hydrologiques", zorder=2)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap)
plt.legend(loc="upper left")
plt.title("Répartition des usines ICPE et des stations hydrologiques dans le département")

### On cherche la quantité d'usines par station

In [ ]:
#On veut le nombre d'usines qui sont dans les 5km de chaque station
rayon = 5000   # En mètres : les géométries sont en Lambert 93 (EPSG:2154), dont l'unité est le mètre

stations_rayon = stations.copy()
stations_rayon["geometry"] = stations_rayon.geometry.buffer(rayon)  # On suppose que la station prend toute la place dans un rayon de 5km autour du point où elle est réellement

usines_ds_rayon = gpd.sjoin(installations_indust, stations_rayon, how="inner", predicate="within")
# Nombre d'usines par station, 0 pour une station sans usine (map au lieu de merge : pas de colonnes nb_usines_5km_x / _y si on relance)
stations["nb_usines_5km"] = stations["code_bss"].map(usines_ds_rayon.groupby("code_bss").size()).fillna(0).astype(int)

#On rajoute une colonne dans le tableau chroniques qui indique la pression industrielle (nb d'usines dans un rayon de 5km qui exploitent probablement la nappe)
chroniques = chroniques.merge(stations[["code_bss", "nb_usines_5km"]], on="code_bss", how="inner")


In [ ]:
# Visualisation
conditions = [
    (chroniques['nb_usines_5km'] == 0),
    (chroniques['nb_usines_5km'] > 0) & (chroniques['nb_usines_5km'] <= 5),
    (chroniques['nb_usines_5km'] > 5)
]
choix = ['0 usine (Zone rurale)', '1 à 5 usines (Intermédiaire)', '> 5 usines (Forte pression)']
chroniques['categorie_industrielle'] = np.select(conditions, choix, default='Inconnu')

evolution_lisse = chroniques.groupby(['date_mois_annee', 'categorie_industrielle'])['niveau_nappe_eau'].mean().unstack()

evolution_lisse.plot(figsize=(12, 6), linewidth=2)
plt.title("Évolution moyenne lissée des nappes selon la pression industrielle", fontsize=14)
plt.ylabel("Niveau d'eau moyen (m)")
plt.xlabel("Date")

On observe que, en moyenne, les nappes qui ont plus de 5 usines à moins de 5km ont un niveau inférieur de 20m.  
Nous n'avons pourtant toujours pas établi de causalité. Le niveau d'une nappe est une altitude (par rapport au niveau de la mer) : cet écart reflète d'abord l'endroit où se trouvent les stations. Les sauts brusques des courbes viennent aussi de ce que l'ensemble des stations mesurées change d'un mois à l'autre. Les conditions géographiques et l'environnement peuvent donc également affecter ces nappes.

In [ ]:
#Marqueur : cette étape est faite, les notebooks qui en dépendent ne la relanceront pas dans ce noyau
ETAPE_02 = True